# Fine-tuning E-RayZer on TartanAir, step by step

This notebook does by hand what `train.py` does in a loop, using the same functions.
Every step shows its result, so you can see what goes into the model, what comes out and which parts learn.

* Run it on a GPU node with the kernel inside the container, for example in the job of `slurm/dev.sh`
* Change model code in `erayzer_core/`, rerun a cell and the change is used without restarting the kernel
* The real run is a Slurm job, see the last section

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, math, time
from pathlib import Path

# Work from the repository root, like train.py does. Inside the container
# you are root, so look upwards from the notebook first, then in the home
# folder the notebook lives in (/mnt/home/<user>/code/P3DCV).
def is_repo(p):
    try:
        return (p / "train.py").is_file() and (p / "erayzer_core").is_dir()
    except OSError:
        return False

cwd = Path.cwd()
candidates = [Path(os.environ.get("REPO_DIR", "/nonexistent")), cwd, *cwd.parents]
if cwd.parts[:3] == ("/", "mnt", "home") and len(cwd.parts) > 3:
    candidates.append(Path("/mnt/home", cwd.parts[3], "code", "P3DCV"))
REPO = next(p for p in candidates if is_repo(p))
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import torch
import matplotlib.pyplot as plt

import train as T   # load_config, build_model, compute_loss, ... from train.py

device = torch.device("cuda")
print(REPO, torch.cuda.get_device_name(0), "bf16", torch.cuda.is_bf16_supported())

## 1. Config

The same YAML file as the Slurm job, with the same `-s key value` overrides.
For the walkthrough we use one small archive, batch size 1 and no worker processes.

In [ ]:
DATA = "/mnt/projects/p3dcv/anycam/data/tartanair"
config = T.load_config("config/erayzer_train_tartanair.yaml", [
    ("dataset.train_path", f"{DATA}/ArchVizTinyHouseNight/Data_hard/image_lcam_fish.zip"),
    ("dataset.camera_name", "lcam_fish"),
    ("training.batch_size_per_gpu", "1"),
    ("training.num_workers", "0"),
    ("training.resume_ckpt", "/mnt/projects/p3dcv/anycam/checkpoints/pretrained/erayzer_multi.pt"),
])
tr = config.training
print("views", tr.num_views, "=", tr.num_input_views, "input +", tr.num_target_views, "target")
print("view selector", dict(tr.view_selector))
print("lr", tr.lr, "warmup", tr.warmup, "loss weights l2", tr.l2_loss_weight, "perceptual", tr.perceptual_loss_weight)

## 2. Data

One sample is a set of 10 frames from one trajectory. The model only uses `image` for training.
`c2w` (ground truth poses) is only used for evaluation, `fxfycxcy` not at all.

In [ ]:
dataset = T.build_config_dataset(config, split="train")
print("samples per epoch", len(dataset))

sample = dataset[0]
for k, v in sample.items():
    print(f"{k:10s}", tuple(v.shape) if torch.is_tensor(v) else v)

In [ ]:
def show_views(images, title="", ncols=5):
    images = images.detach().float().cpu().clamp(0, 1)
    n = images.shape[0]
    fig, axes = plt.subplots(math.ceil(n / ncols), ncols, figsize=(3 * ncols, 3 * math.ceil(n / ncols)))
    for i, ax in enumerate(axes.flat):
        ax.axis("off")
        if i < n:
            ax.imshow(images[i].permute(1, 2, 0))
            ax.set_title(str(i), fontsize=9)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_views(sample["image"], "10 frames of one sample, 256x256 crop of the fisheye")

## 3. Model

`build_model` creates `ERayZer` and loads the pretrained weights.
The parameter groups below are the parts you will most likely change.

In [ ]:
model = T.build_model(config, device, rank=0)
model.train()

groups = {}
for name, p in model.named_parameters():
    groups.setdefault(name.split(".")[0], 0)
    groups[name.split(".")[0]] += p.numel()
for k, n in sorted(groups.items(), key=lambda x: -x[1]):
    print(f"{k:28s} {n / 1e6:8.2f} M")
print("total", sum(groups.values()) / 1e6, "M parameters")

## 4. One forward pass

The model splits the 10 views into 5 input and 5 target views, predicts a camera for every view,
builds Gaussians from the input views and renders them from the predicted target cameras.
`result.render` is compared with `result.target_image`.

In [ ]:
batch = {k: v.unsqueeze(0).to(device) if torch.is_tensor(v) else v for k, v in sample.items()}
loss_modules = T.build_loss_modules(config, device)

with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    result = model(batch, iter=0)
    metrics = T.compute_loss(result, batch, config, loss_modules)

print("input views ", result.input_idx[0].tolist())
print("target views", result.target_idx[0].tolist())
print({k: round(float(v), 4) for k, v in metrics.items()})
show_views(result.target_image[0], "target")
show_views(result.render[0], "render before fine-tuning")

## 5. The camera the model assumes

The model predicts one pinhole camera per view, `fxfycxcy` relative to the image size.
The TartanAir fisheye covers 195 degrees, the pinhole model cannot express that.
This is the place where a learnable distortion would go (`get_cam_se3` and `cam_info_to_plucker` in `erayzer_core/model/erayzer.py`).

In [ ]:
fx = result.fxfycxcy[:, 0].float()
fov = torch.rad2deg(2 * torch.atan(0.5 / fx))
print("fx per view (relative)", [round(x, 3) for x in fx.tolist()])
print("horizontal FOV in degrees", [round(x, 1) for x in fov.tolist()])

## 6. Loss and gradients

Same as one step of `train.py`. The gradient norm per part shows which parts would change.
If you add a parameter, check here that it gets a gradient that is not zero.

In [ ]:
model.zero_grad(set_to_none=True)
with torch.autocast("cuda", dtype=torch.bfloat16):
    result = model(batch, iter=0)
    metrics = T.compute_loss(result, batch, config, loss_modules)
metrics.loss.backward()

norms = {}
for name, p in model.named_parameters():
    if p.grad is not None:
        key = name.split(".")[0]
        norms[key] = norms.get(key, 0.0) + float(p.grad.float().pow(2).sum())
for k, v in sorted(norms.items(), key=lambda x: -x[1]):
    print(f"{k:28s} grad norm {math.sqrt(v):.4f}")
no_grad = [n for n, p in model.named_parameters() if p.requires_grad and p.grad is None]
print(len(no_grad), "parameters without gradient", no_grad[:5])

## 7. A few training steps

Optimizer and learning rate schedule from `train.py`, 30 steps on the same sample.
This is the usual first check after a change: if the model cannot even fit one sample, something is broken.
PSNR has to go up clearly. On new samples every step, batch size 1 is too noisy to see progress,
which is why the real run averages over 8 samples (batch 2 times 4 accumulation steps).

In [ ]:
STEPS = 30
optimizer = T.configure_optimizer(model, float(tr.weight_decay), float(tr.lr), (float(tr.beta1), float(tr.beta2)))
scheduler = T.configure_lr_scheduler(optimizer, STEPS, warmup_steps=5, scheduler_type="cosine")

history = []
t0 = time.time()
for step in range(STEPS):
    b = batch   # same sample every step
    with torch.autocast("cuda", dtype=torch.bfloat16):
        out = model(b, iter=step)
        m = T.compute_loss(out, b, config, loss_modules)
    m.loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), float(tr.grad_clip_norm))
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad(set_to_none=True)
    history.append((float(m.loss), float(m.psnr), float(grad_norm)))
    if step % 5 == 0:
        print(f"step {step:3d} loss {m.loss.item():.4f} psnr {m.psnr.item():.2f} grad {float(grad_norm):.2f} lr {optimizer.param_groups[0]['lr']:.2e}")
print(f"{(time.time() - t0) / STEPS:.2f} s per step, max memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot([h[0] for h in history]); ax[0].set_title("loss")
ax[1].plot([h[1] for h in history]); ax[1].set_title("PSNR")
plt.show()

In [ ]:
with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    result = model(batch, iter=0)
    metrics = T.compute_loss(result, batch, config, loss_modules)
print("same sample after training", {k: round(float(v), 4) for k, v in metrics.items()})
show_views(result.render[0], "render after a few steps")

## 8. The real run

The loop above is what `train.py` runs for 20 000 steps, with gradient accumulation,
checkpoints every 1000 steps and resuming after the 24 hour limit.
Free the GPU first, either restart the kernel or end the dev job, because both need one GPU.

```bash
# on head
cd ~/code/P3DCV
sbatch --export=ALL,RUN_NAME=my-first-run,MAX_STEPS=2000 slurm/train_tartanair.sh
tail -f ~/runs/train-tartanair-<job id>.out      # loss, psnr, lr every 20 steps
ls /mnt/projects/p3dcv/anycam/checkpoints/runs/my-first-run/
```

* All settings of section 1 work the same way, for example `-s training.lr 0.00005` after the script name
* Submitting again with the same `RUN_NAME` continues from the newest checkpoint
* While the dev job runs, a second job needs `--qos=students_opportunistic`

In [ ]:
# Free the GPU memory of this notebook before you start a job on the same GPU.
del model, optimizer, result, out
torch.cuda.empty_cache()